# Barra composita 1D - Metodo de Diferencas Finitas

Use os controles abaixo para ajustar os parametros da simulacao, visualizar os resultados em tela e salvar as saidas quando desejar.

Problema:

-\frac{d}{dx}\left(E(x)A\frac{du}{dx}\right)=0

com (0)=0$ e (L)A u'(L)=T$.

In [ ]:
import base64
import io
import os
import zipfile
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['axes.grid'] = True
plt.rcParams['font.size'] = 11

In [ ]:
@dataclass(frozen=True)
class BarParams:
    length: float = 1.0
    area: float = 1.0
    e1: float = 1.0
    e2: float = 5.0
    traction: float = 1.0


def young_modulus(x, params):
    return np.where(x < params.length / 2.0, params.e1, params.e2)


def element_equivalent_moduli(x, params):
    interface = params.length / 2.0
    moduli = np.empty(len(x) - 1, dtype=float)

    for i, (left, right) in enumerate(zip(x[:-1], x[1:])):
        h = right - left
        if right <= interface:
            moduli[i] = params.e1
        elif left >= interface:
            moduli[i] = params.e2
        else:
            left_length = interface - left
            right_length = right - interface
            moduli[i] = h / (left_length / params.e1 + right_length / params.e2)

    return moduli


def exact_displacement(x, params):
    interface = params.length / 2.0
    strain_1 = params.traction / (params.area * params.e1)
    strain_2 = params.traction / (params.area * params.e2)
    u_interface = strain_1 * interface
    return np.where(
        x <= interface,
        strain_1 * x,
        u_interface + strain_2 * (x - interface),
    )


def relative_l2_error(numerical, reference):
    denominator = np.linalg.norm(reference)
    if denominator == 0.0:
        return float(np.linalg.norm(numerical - reference))
    return float(np.linalg.norm(numerical - reference) / denominator)


def solve_fdm(n, params):
    if n < 2:
        raise ValueError('Use N >= 2.')

    h = params.length / n
    x = np.linspace(0.0, params.length, n + 1)
    e_nodes = young_modulus(x, params)
    e_faces = element_equivalent_moduli(x, params)

    matrix = np.zeros((n + 1, n + 1), dtype=float)
    rhs = np.zeros(n + 1, dtype=float)

    matrix[0, 0] = 1.0

    for i in range(1, n):
        e_left = e_faces[i - 1]
        e_right = e_faces[i]
        matrix[i, i - 1] = -e_left * params.area / h**2
        matrix[i, i] = (e_left + e_right) * params.area / h**2
        matrix[i, i + 1] = -e_right * params.area / h**2

    matrix[n, n - 1] = -e_faces[-1] * params.area / h
    matrix[n, n] = e_faces[-1] * params.area / h
    rhs[n] = params.traction

    u = np.linalg.solve(matrix, rhs)
    x_mid = 0.5 * (x[:-1] + x[1:])
    strain = np.diff(u) / h
    stress = e_faces * strain
    u_exact = exact_displacement(x, params)

    return {
        'x': x,
        'u': u,
        'u_exact': u_exact,
        'x_mid': x_mid,
        'strain': strain,
        'stress': stress,
        'e_nodes': e_nodes,
        'e_faces': e_faces,
        'relative_error': relative_l2_error(u, u_exact),
    }


def effective_modulus_series(params):
    l1 = params.length / 2.0
    l2 = params.length / 2.0
    return params.length / (l1 / params.e1 + l2 / params.e2)

In [ ]:
def result_tables(result):
    nodal = pd.DataFrame({
        'x': result['x'],
        'E(x)': result['e_nodes'],
        'u_MDF': result['u'],
        'u_analitico': result['u_exact'],
        'erro_abs': np.abs(result['u'] - result['u_exact']),
    })

    elemental = pd.DataFrame({
        'x_centro': result['x_mid'],
        'E_equiv': result['e_faces'],
        'epsilon': result['strain'],
        'sigma': result['stress'],
    })

    return nodal, elemental


def plot_result(result, params):
    fig, axes = plt.subplots(3, 1, sharex=False, figsize=(12, 9))
    interface = params.length / 2.0

    axes[0].plot(result['x'], result['u'], 'o-', label='MDF')
    axes[0].plot(result['x'], result['u_exact'], '--', label='Analitica')
    axes[0].set_ylabel('u(x)')
    axes[0].legend()

    axes[1].step(result['x_mid'], result['strain'], where='mid', color='tab:orange')
    axes[1].set_ylabel('epsilon(x)')

    axes[2].step(result['x_mid'], result['stress'], where='mid', color='tab:green')
    axes[2].set_ylabel('sigma(x)')
    axes[2].set_xlabel('x')

    for ax in axes:
        ax.axvline(interface, color='k', linestyle=':', linewidth=1.4, label='Interface')

    fig.tight_layout()
    plt.show()


def plot_result_as_base64(result, params):
    fig, axes = plt.subplots(3, 1, sharex=False, figsize=(11.5, 8.2))
    interface = params.length / 2.0

    axes[0].plot(result['x'], result['u'], 'o-', label='MDF')
    axes[0].plot(result['x'], result['u_exact'], '--', label='Analitica')
    axes[0].set_ylabel('u(x)')
    axes[0].legend(loc='upper left')

    axes[1].step(result['x_mid'], result['strain'], where='mid', color='tab:orange')
    axes[1].set_ylabel('epsilon(x)')

    axes[2].step(result['x_mid'], result['stress'], where='mid', color='tab:green')
    axes[2].set_ylabel('sigma(x)')
    axes[2].set_xlabel('x')

    for ax in axes:
        ax.axvline(interface, color='k', linestyle=':', linewidth=1.4)

    fig.suptitle('Barra composita 1D - Metodo de Diferencas Finitas')
    fig.tight_layout()

    buffer = io.BytesIO()
    fig.savefig(buffer, format='png', dpi=130, bbox_inches='tight')
    plt.close(fig)
    return base64.b64encode(buffer.getvalue()).decode('ascii')


def save_outputs(result, params, output_dir='resultados_colab'):
    os.makedirs(output_dir, exist_ok=True)
    nodal, elemental = result_tables(result)
    nodal.to_csv(os.path.join(output_dir, 'deslocamentos.csv'), index=False)
    elemental.to_csv(os.path.join(output_dir, 'deformacao_tensao.csv'), index=False)

    with open(os.path.join(output_dir, 'resumo.txt'), 'w', encoding='utf-8') as f:
        f.write('Projeto 1 - Barra composita 1D pelo MDF\n')
        f.write(f'L = {params.length}\n')
        f.write(f'A = {params.area}\n')
        f.write(f'E1 = {params.e1}\n')
        f.write(f'E2 = {params.e2}\n')
        f.write(f'T = {params.traction}\n')
        f.write(f'u(L) = {result["u"][-1]}\n')
        f.write(f'Erro relativo L2 = {result["relative_error"]}\n')
        f.write(f'Eef serie = {effective_modulus_series(params)}\n')

    zip_path = f'{output_dir}.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for name in os.listdir(output_dir):
            zf.write(os.path.join(output_dir, name), arcname=name)

    return output_dir, zip_path

In [ ]:
def run_interactive(L, A, E1, E2, T, N, salvar_saidas, baixar_zip):
    params = BarParams(length=L, area=A, e1=E1, e2=E2, traction=T)
    result = solve_fdm(N, params)
    nodal, elemental = result_tables(result)

    plot_png = plot_result_as_base64(result, params)
    nodal_html = nodal.head(8).to_html(index=False, classes='result-table')
    elemental_html = elemental.head(8).to_html(index=False, classes='result-table')
    summary = f'''Resumo da simulacao\nN = {N}\nE2/E1 = {E2 / E1:.6g}\nu(L) = {result["u"][-1]:.10g}\nErro relativo L2 = {result["relative_error"]:.3e}\nEef serie = {effective_modulus_series(params):.10g}\nsigma esperado = T/A = {T / A:.10g}'''

    display(HTML(f'''
    <style>
      .p1-output {{
        display: grid;
        grid-template-columns: minmax(620px, 1.45fr) minmax(460px, 1fr);
        gap: 28px;
        align-items: start;
        width: 100%;
      }}
      .p1-panel {{ border: 4px solid #3f46d9; background: white; }}
      .p1-plot {{ padding: 46px 58px 54px 58px; }}
      .p1-section-title {{ margin: 0 0 22px 0; font-size: 18px; font-weight: 700; }}
      .p1-plot img {{ width: 100%; height: auto; display: block; }}
      .p1-right {{ display: flex; flex-direction: column; }}
      .p1-summary {{
        padding: 34px 40px 36px 40px;
        border-bottom: 4px solid #3f46d9;
        font-family: Consolas, 'Courier New', monospace;
        font-size: 17px;
        line-height: 1.35;
        white-space: pre-wrap;
      }}
      .p1-tables {{ display: flex; flex-direction: column; gap: 34px; padding: 52px 58px; }}
      .p1-tables h4 {{ margin: 0 0 10px 0; font-size: 14px; font-weight: 700; }}
      .result-table {{ border-collapse: collapse; font-size: 12px; width: 100%; }}
      .result-table th {{ text-align: right; padding: 6px 8px; border-bottom: 1px solid #bbb; }}
      .result-table td {{ text-align: right; padding: 6px 8px; }}
      .result-table tbody tr:nth-child(odd) {{ background: #f2f2f2; }}
      @media (max-width: 980px) {{ .p1-output {{ grid-template-columns: 1fr; }} }}
    </style>
    <div class="p1-output">
      <div class="p1-panel p1-plot">
        <div class="p1-section-title">Campos calculados</div>
        <img src="data:image/png;base64,{plot_png}" alt="Graficos da barra composita">
      </div>
      <div class="p1-panel p1-right">
        <div class="p1-summary">{summary}</div>
        <div class="p1-tables">
          <div>
            <h4>Primeiras linhas dos deslocamentos nodais</h4>
            {nodal_html}
          </div>
          <div>
            <h4>Primeiras linhas de deformacao e tensao</h4>
            {elemental_html}
          </div>
        </div>
      </div>
    </div>
    '''))

    if salvar_saidas:
        output_dir, zip_path = save_outputs(result, params)
        print(f'Saidas salvas em: {output_dir}')
        print(f'Arquivo zip gerado: {zip_path}')
        if baixar_zip and IN_COLAB:
            files.download(zip_path)


controls = {
    'L': widgets.FloatSlider(value=1.0, min=0.2, max=5.0, step=0.1, description='L'),
    'A': widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='A'),
    'E1': widgets.FloatLogSlider(value=1.0, base=10, min=-1, max=2, step=0.05, description='E1'),
    'E2': widgets.FloatLogSlider(value=5.0, base=10, min=-1, max=2, step=0.05, description='E2'),
    'T': widgets.FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, description='T'),
    'N': widgets.IntSlider(value=40, min=4, max=300, step=2, description='N'),
    'salvar_saidas': widgets.Checkbox(value=False, description='Salvar saidas'),
    'baixar_zip': widgets.Checkbox(value=False, description='Baixar zip'),
}

ui_left = widgets.VBox(
    [controls['L'], controls['A'], controls['T'], controls['N']],
    layout=widgets.Layout(gap='8px'),
)
ui_right = widgets.VBox(
    [controls['E1'], controls['E2'], controls['salvar_saidas'], controls['baixar_zip']],
    layout=widgets.Layout(gap='8px'),
)
ui = widgets.HBox(
    [ui_left, ui_right],
    layout=widgets.Layout(justify_content='center', gap='72px', margin='10px 0 56px 0'),
)
out = widgets.interactive_output(run_interactive, controls)

display(HTML('''
<div style="text-align:center; margin: 18px 0 34px 0;">
  <h1 style="margin:0; font-size:28px; font-weight:600;">Barra composita 1D - Metodo de Diferencas Finitas</h1>
</div>
'''))
display(ui, out)

Output()

## Validacao sugerida pelo PDF

Para validar contra o caso apresentado no PDF, ajuste os controles para:

- `L = 1`
- `A = 1`
- `T = 1`
- `E1 = 1`
- `E2 = 10`

Os resultados esperados sao:

- `u(L) = 0.55`
- `sigma = 1`
- `epsilon1 = 1`
- `epsilon2 = 0.1`
- `Eef = 1.81818`